# 1. Domain Adaptation (Task Adaptation)

In [1]:
# ============================================================
# 1: Verify Kaggle environment
# ============================================================

import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))

    # Simple CUDA test
    x = torch.tensor([1.0, 2.0, 3.0]).cuda()
    print("CUDA test:", x)

PyTorch: 2.10.0+cu128
CUDA: 12.8
CUDA available: True
GPU: Tesla T4
Compute capability: (7, 5)
CUDA test: tensor([1., 2., 3.], device='cuda:0')


In [2]:
# ============================================================
# 2: Imports
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    DataCollatorForLanguageModeling,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

In [3]:
# ============================================================
# 3: Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

Random seed: 42


In [4]:
# ============================================================
# 4: Experiment configuration
# ============================================================

MODEL_NAME = "FacebookAI/xlm-roberta-base"

TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

NUM_LABELS = 2

MAX_LENGTH = 128

# DAPT
DAPT_OUTPUT_DIR = "/kaggle/working/xlm_r_dapt"
DAPT_FINAL_DIR = "/kaggle/working/xlm_r_dapt_final"

# Classification
CLASSIFICATION_OUTPUT_DIR = "/kaggle/working/xlm_r_dapt_classifier"
CLASSIFICATION_FINAL_DIR = "/kaggle/working/xlm_r_dapt_classifier_final"

print("Model:", MODEL_NAME)
print("Max length:", MAX_LENGTH)
print("Number of labels:", NUM_LABELS)

Model: FacebookAI/xlm-roberta-base
Max length: 128
Number of labels: 2


In [5]:
# ============================================================
# 5: Dataset paths
# ============================================================

TRAIN_PATH = "/kaggle/input/datasets/uom220676x/clean-sinhala-train-and-test/clean_sinhala_train.jsonl"
TEST_PATH = "/kaggle/input/datasets/uom220676x/clean-sinhala-train-and-test/clean_sinhala_test.jsonl"

In [6]:
# ============================================================
# 6: Load dataset
# ============================================================

train_df = pd.read_json(
    TRAIN_PATH,
    lines=True
)

test_df = pd.read_json(
    TEST_PATH,
    lines=True
)

print("Original training set:", train_df.shape)
print("Original test set:", test_df.shape)

display(train_df.head())

Original training set: (7500, 6)
Original test set: (2500, 6)


,post_id,text,tokens,rationales,label,split
0,726758237668659200,@USER @USER පට්ට පට පට...,@USER @USER පට්ට පට පට . . .,[],0,train
1,915618589855617024,පරණ කෑල්ල අද වෙනකම් හිටියනම් අදට අවුරුදු 4යි. ...,පරණ කෑල්ල අද වෙනකම් හිටියනම් අදට අවුරුදු 4යි ....,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,train
2,925001070430040064,යාළුවා කියලා හිතන් සර් ගේ ඔලුවට රෙද්ද දාලා නෙල...,යාළුවා කියලා හිතන් සර් ගේ ඔලුවට රෙද්ද දාලා නෙල...,[],0,train
3,1397219745707986944,හොඳ මිතුරියක් කතා කලා. විස්තර කතාකරමින් ඉදලා ම...,හොඳ මිතුරියක් කතා කලා . විස්තර කතාකරමින් ඉදලා ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,train
4,950376113150222336,"ඔය බනින්නෙ.. හරකා, මී හරකා කිය කිය...","ඔය බනින්නෙ . . හරකා , මී හරකා කිය කිය . . .","[0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0]",1,train


In [7]:
# ============================================================
# 7: Prepare dataframe
# ============================================================

train_df = train_df[
    [TEXT_COLUMN, LABEL_COLUMN]
].copy()

test_df = test_df[
    [TEXT_COLUMN, LABEL_COLUMN]
].copy()

# Remove missing values.
train_df = train_df.dropna()
test_df = test_df.dropna()

# Ensure text is string.
train_df[TEXT_COLUMN] = train_df[TEXT_COLUMN].astype(str)
test_df[TEXT_COLUMN] = test_df[TEXT_COLUMN].astype(str)

# Ensure labels are native Python integers.
train_df[LABEL_COLUMN] = train_df[LABEL_COLUMN].astype(int)
test_df[LABEL_COLUMN] = test_df[LABEL_COLUMN].astype(int)

print("Training:", train_df.shape)
print("Test:", test_df.shape)

Training: (7500, 2)
Test: (2500, 2)


In [8]:
# ============================================================
# 8: Create validation set
# ============================================================

train_df, valid_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=SEED,
    stratify=train_df[LABEL_COLUMN]
)

# Reset indices.
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

train_df.head()

Train: (6750, 2)
Validation: (750, 2)
Test: (2500, 2)


,text,label
0,@USER @USER පිංගොනා අහනෝ තෝ වැස්සි ඔන්ලයින් ඩ...,1
1,@USER අනේ උඹේ අම්මට හුකාපං පොන්න වේසිගෙ පුතා. ...,1
2,@USER තෝ කොළඹ යනවා කිව්වා නේද හුත්තෝ,1
3,@USER @USER තිබ්බත් සල්ලි වියදම් කරන්ඩ එපා...,0
4,"රටේ දරුවො ගැන නොහිතන කාලකණ්ණි පාලකයෝ., පටු වාස...",1


In [9]:
# ============================================================
# 9: Check class distributions
# ============================================================

print("TRAIN")
print(train_df[LABEL_COLUMN].value_counts(normalize=True))

print("\nVALIDATION")
print(valid_df[LABEL_COLUMN].value_counts(normalize=True))

print("\nTEST")
print(test_df[LABEL_COLUMN].value_counts(normalize=True))

TRAIN
label
0    0.576593
1    0.423407
Name: proportion, dtype: float64

VALIDATION
label
0    0.576
1    0.424
Name: proportion, dtype: float64

TEST
label
0    0.594
1    0.406
Name: proportion, dtype: float64


In [10]:
# ============================================================
# 10: Label mappings
# ============================================================

label2id = {
    "non-hate": 0,
    "hate": 1,
}

id2label = {
    0: "non-hate",
    1: "hate",
}

# Create the "labels" column used by the classifier.
train_df["labels"] = train_df[LABEL_COLUMN]
valid_df["labels"] = valid_df[LABEL_COLUMN]
test_df["labels"] = test_df[LABEL_COLUMN]

print("label2id:", label2id)
print("id2label:", id2label)

label2id: {'non-hate': 0, 'hate': 1}
id2label: {0: 'non-hate', 1: 'hate'}


In [11]:
# ============================================================
# 11: Load original XLM-R tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocabulary size: 250002


In [12]:
# ============================================================
# 12: Create DAPT dataset
# ============================================================

dapt_train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COLUMN]],
    preserve_index=False
)

print(dapt_train_dataset)

Dataset({
    features: ['text'],
    num_rows: 6750
})


In [13]:
# ============================================================
# 13: Tokenize DAPT corpus
# ============================================================

def tokenize_for_dapt(examples):

    return tokenizer(
        examples[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH,
    )


dapt_tokenized = dapt_train_dataset.map(
    tokenize_for_dapt,
    batched=True,
    remove_columns=[TEXT_COLUMN],
)

print(dapt_tokenized)

Map:   0%|          | 0/6750 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 6750
})


In [14]:
# ============================================================
# 14: MLM data collator
# ============================================================

mlm_data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

print("MLM data collator created.")

MLM data collator created.


In [15]:
# ============================================================
# 15: Load XLM-R for masked language modeling
# ============================================================

dapt_model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME
)

print("XLM-R MLM model loaded.")

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

XLMRobertaForMaskedLM LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLM-R MLM model loaded.


In [16]:
# ============================================================
# 16: DAPT training configuration
# ============================================================

dapt_training_args = TrainingArguments(

    output_dir=DAPT_OUTPUT_DIR,

    # Start with a small number of epochs because
    # the corpus contains only 6,750 texts.
    num_train_epochs=3,

    # Adjust if GPU memory is limited.
    per_device_train_batch_size=8,

    # Gradient accumulation gives an effective
    # larger batch size without requiring as much GPU memory.
    gradient_accumulation_steps=2,

    # Learning rate for continued MLM pretraining.
    learning_rate=5e-5,

    # Regularization.
    weight_decay=0.01,

    # Logging.
    logging_strategy="steps",
    logging_steps=50,

    # Save checkpoints.
    save_strategy="epoch",
    save_total_limit=2,

    # Mixed precision on compatible GPU.
    fp16=torch.cuda.is_available(),

    # Reproducibility.
    seed=SEED,

    report_to="none",
)

In [17]:
# ============================================================
# 17: Create DAPT Trainer
# ============================================================

dapt_trainer = Trainer(
    model=dapt_model,
    args=dapt_training_args,
    train_dataset=dapt_tokenized,
    processing_class=tokenizer,
    data_collator=mlm_data_collator,
)

print("DAPT Trainer created.")

DAPT Trainer created.


In [18]:
# ============================================================
# 18: Perform domain-adaptive pretraining
# ============================================================

dapt_result = dapt_trainer.train()

print("\nDAPT completed.")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,12.664569
100,12.179683
150,11.878580
200,11.803774
250,11.542803
300,11.024990
350,11.181228
400,10.779287
450,10.977950
500,10.805988


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


DAPT completed.


In [19]:
# ============================================================
# 19: Save domain-adapted model
# ============================================================

dapt_trainer.save_model(
    DAPT_FINAL_DIR
)

tokenizer.save_pretrained(
    DAPT_FINAL_DIR
)

print("DAPT model saved to:")
print(DAPT_FINAL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DAPT model saved to:
/kaggle/working/xlm_r_dapt_final


# 2. Classification Training

In [20]:
# ============================================================
# 20: Load DAPT model for classification
# ============================================================

classification_model = AutoModelForSequenceClassification.from_pretrained(
    DAPT_FINAL_DIR,

    num_labels=NUM_LABELS,

    id2label=id2label,
    label2id=label2id,
)

print("DAPT model loaded for classification.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: /kaggle/working/xlm_r_dapt_final
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DAPT model loaded for classification.


In [21]:
# ============================================================
# 21: Enable full fine-tuning
# ============================================================

for param in classification_model.parameters():
    param.requires_grad = True


total_params = sum(
    p.numel()
    for p in classification_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in classification_model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(
    f"Trainable percentage: "
    f"{100 * trainable_params / total_params:.2f}%"
)

Total parameters:     278,045,186
Trainable parameters: 278,045,186
Trainable percentage: 100.00%


In [22]:
# ============================================================
# 22: Create classification datasets
# ============================================================

classification_dataset = DatasetDict({

    "train": Dataset.from_pandas(
        train_df[[TEXT_COLUMN, "labels"]],
        preserve_index=False
    ),

    "validation": Dataset.from_pandas(
        valid_df[[TEXT_COLUMN, "labels"]],
        preserve_index=False
    ),

    "test": Dataset.from_pandas(
        test_df[[TEXT_COLUMN, "labels"]],
        preserve_index=False
    ),
})

print(classification_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 6750
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 750
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 2500
    })
})


In [23]:
# ============================================================
# 23: Tokenize classification data
# ============================================================

def tokenize_classification(examples):

    return tokenizer(
        examples[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH,
    )


classification_tokenized = classification_dataset.map(
    tokenize_classification,
    batched=True,
    remove_columns=[TEXT_COLUMN],
)

print(classification_tokenized)

Map:   0%|          | 0/6750 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 6750
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 750
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2500
    })
})


In [24]:
# ============================================================
# 24: Classification data collator
# ============================================================

classification_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Classification data collator created.")

Classification data collator created.


In [25]:
# ============================================================
# 25: Evaluation metrics
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, macro_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    _, _, weighted_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }

In [26]:
# ============================================================
# CELL 26: Classification fine-tuning configuration
# ============================================================

classification_training_args = TrainingArguments(

    output_dir=CLASSIFICATION_OUTPUT_DIR,

    # Start with 5 epochs.
    num_train_epochs=8,

    # Adjust according to GPU memory.
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    # Full XLM-R fine-tuning requires a small LR.
    learning_rate=2e-5,

    weight_decay=0.01,

    # Evaluate after every epoch.
    eval_strategy="epoch",

    # Save after every epoch.
    save_strategy="epoch",

    # Keep the best validation model.
    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",
    greater_is_better=True,

    logging_strategy="steps",
    logging_steps=50,

    save_total_limit=2,

    fp16=torch.cuda.is_available(),

    seed=SEED,

    report_to="none",
)

In [27]:
# ============================================================
# 27: Create classification Trainer
# ============================================================

classification_trainer = Trainer(

    model=classification_model,
    args=classification_training_args,
    train_dataset=classification_tokenized["train"],
    eval_dataset=classification_tokenized["validation"],
    processing_class=tokenizer,
    data_collator=classification_collator,
    compute_metrics=compute_metrics,
)

print("Classification Trainer created.")

Classification Trainer created.


In [28]:
# ============================================================
# 28: Downstream classification fine-tuning
# ============================================================

classification_train_result = (
    classification_trainer.train()
)

print("\nDownstream fine-tuning completed.")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
1,1.320535,1.321790,0.594667,0.594635,0.534460,0.475061,0.513148
2,0.999708,0.860010,0.826667,0.822697,0.822152,0.822419,0.826593
3,0.837304,0.789487,0.834667,0.833034,0.840715,0.833376,0.835605
4,0.714690,0.713219,0.844000,0.839790,0.842593,0.841001,0.844320
5,0.583823,0.730040,0.849333,0.845176,0.849712,0.846877,0.849825
6,0.503471,0.809417,0.842667,0.838516,0.840605,0.839460,0.842909
7,0.434185,0.843589,0.849333,0.845437,0.846807,0.846082,0.849482
8,0.401159,0.882762,0.846667,0.843295,0.842418,0.842842,0.846569


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


... (9 outputs left out)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Downstream fine-tuning completed.


In [29]:
# ============================================================
# 29: Validation evaluation
# ============================================================

validation_results = classification_trainer.evaluate(
    classification_tokenized["validation"]
)

print("Validation results:")

for key, value in validation_results.items():
    print(f"{key}: {value}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Validation results:
eval_loss: 0.7306482195854187
eval_accuracy: 0.8506666666666667
eval_macro_precision: 0.8465105244996549
eval_macro_recall: 0.8508691474493362
eval_macro_f1: 0.8481792352571194
eval_weighted_f1: 0.8511330600559568
eval_runtime: 3.8645
eval_samples_per_second: 194.076
eval_steps_per_second: 3.105
epoch: 8.0


# 3. Evaluation

In [30]:
# ============================================================
# 30: Final test evaluation
# ============================================================

test_results = classification_trainer.evaluate(
    classification_tokenized["test"]
)

print("Final test results:")

for key, value in test_results.items():
    print(f"{key}: {value}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Final test results:
eval_loss: 0.8493123054504395
eval_accuracy: 0.8324
eval_macro_precision: 0.825687906182303
eval_macro_recall: 0.8319500422948699
eval_macro_f1: 0.8280642197784088
eval_weighted_f1: 0.8331972593354563
eval_runtime: 12.899
eval_samples_per_second: 193.814
eval_steps_per_second: 3.101
epoch: 8.0


In [31]:
# ============================================================
# 31: Detailed classification report
# ============================================================

predictions = classification_trainer.predict(
    classification_tokenized["test"]
)

predicted_labels = np.argmax(
    predictions.predictions,
    axis=-1
)

true_labels = predictions.label_ids

print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=[
            id2label[i]
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

    non-hate     0.8775    0.8343    0.8554      1485
        hate     0.7739    0.8296    0.8008      1015

    accuracy                         0.8324      2500
   macro avg     0.8257    0.8320    0.8281      2500
weighted avg     0.8354    0.8324    0.8332      2500



In [32]:
# ============================================================
# 32: Confusion matrix
# ============================================================

cm = confusion_matrix(
    true_labels,
    predicted_labels
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[1239  246]
 [ 173  842]]


In [33]:
# ============================================================
# 33: Save final classification model
# ============================================================

classification_trainer.save_model(
    CLASSIFICATION_FINAL_DIR
)

tokenizer.save_pretrained(
    CLASSIFICATION_FINAL_DIR
)

print("Final model saved to:")
print(CLASSIFICATION_FINAL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved to:
/kaggle/working/xlm_r_dapt_classifier_final


In [34]:
# ============================================================
# 34: Save experiment results
# ============================================================

experiment_results = {

    "experiment":
        "XLM-R DAPT + Full Fine-Tuning",

    "base_model":
        MODEL_NAME,

    "tokenizer":
        "Original XLM-R tokenizer",

    "dapt":
        True,

    "dapt_corpus_size":
        len(train_df),

    "classification_train_size":
        len(train_df),

    "validation_size":
        len(valid_df),

    "test_size":
        len(test_df),

    "dapt_epochs":
        dapt_training_args.num_train_epochs,

    "dapt_learning_rate":
        dapt_training_args.learning_rate,

    "classification_epochs":
        classification_training_args.num_train_epochs,

    "classification_learning_rate":
        classification_training_args.learning_rate,

    "max_length":
        MAX_LENGTH,

    "seed":
        SEED,

    "accuracy":
        test_results.get("eval_accuracy"),

    "macro_precision":
        test_results.get("eval_macro_precision"),

    "macro_recall":
        test_results.get("eval_macro_recall"),

    "macro_f1":
        test_results.get("eval_macro_f1"),

    "weighted_f1":
        test_results.get("eval_weighted_f1"),
}

results_df = pd.DataFrame(
    [experiment_results]
)

display(results_df)

results_df.to_csv(
    "/kaggle/working/xlm_r_dapt_results.csv",
    index=False
)

print("Results saved.")

,experiment,base_model,tokenizer,dapt,dapt_corpus_size,classification_train_size,validation_size,test_size,dapt_epochs,dapt_learning_rate,classification_epochs,classification_learning_rate,max_length,seed,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
0,XLM-R DAPT + Full Fine-Tuning,FacebookAI/xlm-roberta-base,Original XLM-R tokenizer,True,6750,6750,750,2500,3,0.00005,8,0.00002,128,42,0.8324,0.825688,0.83195,0.828064,0.833197


Results saved.
